In [ ]:
# Train a student ReLU MLP network on the following dataset:
# Input distribution is x \in \R^d drawn as a sum of t_i * a_i for i \in S, where S is a random subset of {1,...,m} of size k, a_i are fixed random vectors in \R^d, and t_i are i.i.d. normal random variables.
# Labels are \sum_{i \in S} z_i * |t_i|, where z_i are fixed random signs.
import argparse
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
class RandomSubsetDataset(Dataset):
    def __init__(self, m, d, k, num_samples):
        self.m = m
        self.d = d
        self.k = k
        self.num_samples = num_samples
        self.a = np.random.randn(m, d)
        self.z = np.random.choice([-1, 1], size=m)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        S = np.random.choice(self.m, self.k, replace=False)
        t = np.random.randn(self.k)
        x = np.sum(t[:, None] * self.a[S], axis=0)
        label = np.sum(self.z[S] * np.abs(t))
        return torch.tensor(x, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

def train_model(model, dataloader, num_epochs, learning_rate):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in dataloader:
            inputs = inputs.to(model.fc1.weight.device)
            labels = labels.to(model.fc2.weight.device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()

        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

def run_experiment(args):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    torch.manual_seed(0)
    np.random.seed(0)
    if device.type == 'cuda':
        torch.cuda.manual_seed(0)
    print(f'Setting random seed to 0 for reproducibility')
    dataset = RandomSubsetDataset(args.m, args.d, args.k, args.num_samples)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True)

    print(f'Created dataset with {args.num_samples} samples, each of dimension {args.d}, using {args.m} vectors and subset size {args.k}')
    print(f'Batch size: {args.batch_size}, Number of epochs: {args.num_epochs}, Learning rate: {args.learning_rate}')
    print(f'Hidden dimension: {args.hidden_dim}')
    print(f'Saving model: {args.save_model}, Model path: {args.model_path}') 

    model = MLP(args.d, args.hidden_dim, 1)
    model.to(device)
    train_model(model, dataloader, args.num_epochs, args.learning_rate)

    if args.save_model:
        torch.save(model.state_dict(), args.model_path)
        print(f'Model saved to {args.model_path}')


parser = argparse.ArgumentParser()
parser.add_argument('--m', type=int, default=100)
parser.add_argument('--d', type=int, default=10)
parser.add_argument('--k', type=int, default=5)
parser.add_argument('--num_samples', type=int, default=1000)
parser.add_argument('--batch_size', type=int, default=32)
parser.add_argument('--num_epochs', type=int, default=10)
parser.add_argument('--learning_rate', type=float, default=1e-3)
parser.add_argument('--hidden_dim', type=int, default=64)
parser.add_argument('--save_model', action='store_true')
parser.add_argument('--model_path', type=str, default='model.pth')
args = parser.parse_args(['--k', '80', '--m', '10000', '--d', '512', '--num_samples', '100000', '--batch_size', '32', '--num_epochs', '10', '--learning_rate', '0.0001', '--hidden_dim', '512'])
run_experiment(args)

Using device: cuda
Setting random seed to 0 for reproducibility
Created dataset with 100000 samples, each of dimension 512, using 10000 vectors and subset size 80
Batch size: 32, Number of epochs: 10, Learning rate: 1e-05
Hidden dimension: 512
Saving model: False, Model path: model.pth
Epoch [1/10], Loss: 52.7046
